# 32 — Final test evaluation: classical champions

**Protocol (the sanctioned, one-shot use of test).** Configurations were frozen on **dev** in the
benchmark notebooks; nothing here is chosen by looking at test. The selected sentiment champions
are refit on **train+dev** (all 9,998 ids in the official train file, fanned to five languages)
and scored **once** on the official held-out test set.

`headline` for sentiment is **Negative-F1**, never accuracy (the Negative class is rare —
~3.3% prevalence on test).

**Why test and not just dev.** Dev holds only ~340 pooled Negative tickets; test holds **505**,
so the Negative-F1 estimate is tighter. That is the whole reason to open test here — at the cost
of test no longer being an unbiased *future* estimate.

Companion notebook: [`33_final_test_encoders.ipynb`](33_final_test_encoders.ipynb) does the same
for the encoder roster (trained on Kaggle T4s).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, data, imbalance, models, splits, metrics, results

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

TASK = "sentiment"
LANGS = list(config.LANGUAGES)
LABEL = data.label_column(TASK)
TEXT = config.TEXT_COLUMN
print("split sha:", splits.sha())

split sha: e7b5934392cd


In [2]:
# Frozen on dev, before test was opened (see 03/06/10). (model, arm, C).
SENTIMENT_CONFIGS = [("tfidf-svm", "class_weight", 0.5),
                     ("tfidf-logreg", "ros", 3.0)]

TRAIN_ONLY = splits.get(LANGS, "train")            # 42,500 — reproduces the dev number
TRAIN_DEV  = data.load_languages(LANGS, "train")   # 49,990 — every id in the official train file
TEST       = splits.get(LANGS, "test")             # 15,395 — official BANKING77 test, 505 Negative
assert len(TRAIN_DEV) == 9998 * len(LANGS), len(TRAIN_DEV)
print(f"train {len(TRAIN_ONLY):,} | train+dev {len(TRAIN_DEV):,} | test {len(TEST):,}")
print("test Negative count:", int((TEST[LABEL] == 'Negative').sum()))

train 42,500 | train+dev 49,990 | test 15,395


test Negative count: 505


In [3]:
def fit_eval(model, arm, C, train_df, eval_df):
    fit = imbalance.resample(train_df, LABEL, arm)
    clf = models.build(model, class_weight=imbalance.class_weight_for(arm), C=C)
    clf.fit(fit[TEXT], fit[LABEL])
    pred = clf.predict(eval_df[TEXT])
    return metrics.score(eval_df[LABEL], pred, TASK)

rows = []
for model, arm, C in SENTIMENT_CONFIGS:
    dev = fit_eval(model, arm, C, TRAIN_ONLY, splits.get(LANGS, "dev"))   # fit train -> dev
    tst = fit_eval(model, arm, C, TRAIN_DEV, TEST)                        # fit train+dev -> test
    results.save(TASK, model, LANGS, "all", arm, "test", tst, author="final-test-classical")
    rows.append({"model": model, "arm": arm, "C": C,
                 "dev_negF1": dev["negative_f1"], "test_negF1": tst["negative_f1"],
                 "drop": dev["negative_f1"] - tst["negative_f1"],
                 "test_macroF1": tst["macro_f1"], "test_acc": tst["accuracy"]})
res = pd.DataFrame(rows).sort_values("test_negF1", ascending=False)
display(res)
print(f"\nClassical bar to beat on test (best Negative-F1): {res.test_negF1.max():.4f}")

,model,arm,C,dev_negF1,test_negF1,drop,test_macroF1,test_acc
0,tfidf-svm,class_weight,0.5000,0.6187,0.4635,0.1551,0.7191,0.9517
1,tfidf-logreg,ros,3.0000,0.5945,0.4498,0.1447,0.7126,0.9530



Classical bar to beat on test (best Negative-F1): 0.4635


## Reading this

- **Expect a large dev→test drop.** It reproduces the project's standing finding: the sentiment
  champion loses ~0.15 Negative-F1 from dev to test. Part is test's lower Negative prevalence
  (3.3% vs 4.5%), the rest is genuine generalization loss.
- The **best classical test Negative-F1** printed above is the production number the encoders must
  clear in [`33_final_test_encoders.ipynb`](33_final_test_encoders.ipynb).
- Test JSONs are saved to `ml/reports/runs/` (`author=final-test-classical`, `ev-all`, `test`) so
  the combined leaderboard in notebook 33 can read them.